# 🧠 การกำหนดค่า Learning Rate (Scheduling) และ Cosine Annealing

ยินดีต้อนรับสู่สมุดบันทึกคำอธิบายเชิงปฏิบัติสำหรับ **Learning Rate Scheduling**! ในสมุดบันทึกนี้ เราจะ:
1. กำหนดสูตรทางคณิตศาสตร์สำหรับตัวกำหนดค่า (schedulers) แบบ Linear Warmup, Step Decay และ Cosine Annealing
2. เขียนโค้ดเพื่อสร้างตัวกำหนดค่าเหล่านี้ขึ้นมาจากศูนย์ด้วยภาษา Python
3. พล็อตและเปรียบเทียบกราฟการปรับเปลี่ยนของตัวกำหนดค่าตลอด 100 epochs ในการฝึกสอน (สไตล์ YOLO)
4. จำลองการทำ Gradient Descent แบบ 2 มิติ บนฟังก์ชัน $f(x,y) = x^2 + 3y^2$ เพื่อเปรียบเทียบระหว่าง **Constant Learning Rate** กับ **Cosine Annealing Learning Rate**
5. พล็อตเส้นทางการหาค่าเหมาะที่สุด (optimization trajectories) บนแผนที่คอนทัวร์แบบ 2 มิติ เพื่อสังเกตว่าการค่อยๆ ลด learning rate ช่วยรักษาความเสถียรในการลู่เข้าสู่จุดต่ำสุดรวม (global minimum) ได้อย่างไร
6. เชื่อมโยงการกำหนดค่าเหล่านี้เข้ากับไฮเปอร์พารามิเตอร์เริ่มต้นสำหรับการฝึกสอนของ YOLO

มาเริ่มต้นด้วยการนำเข้าไลบรารีที่จำเป็นกันก่อน

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Set seed for reproducibility
np.random.seed(42)

## 1. การสร้าง Schedulers ขึ้นมาจากศูนย์

มาเขียนฟังก์ชันกำหนดค่าการเรียนรู้ที่เรากำหนดเองกัน:
-   **Step Decay:** ลดค่า learning rate ลงตามตัวคูณ (เช่น 0.5) ทุกๆ 20 epochs
-   **Cosine Annealing with Warmup:** เพิ่มค่า learning rate แบบเชิงเส้น (warmup) ในช่วง 10 epochs แรก จากนั้นค่อยๆ ลดลงตามโค้งโคไซน์ (cosine curve) จนถึงเป้าหมายขั้นต่ำ

In [ ]:
def step_decay(epoch, lr_initial=0.1, step_size=20, gamma=0.5):
    return lr_initial * (gamma ** (epoch // step_size))

def cosine_annealing_with_warmup(epoch, total_epochs=100, warmup_epochs=10, lr_max=0.1, lr_min=0.001):
    if epoch < warmup_epochs:
        # Linear Warmup
        return lr_min + (epoch / warmup_epochs) * (lr_max - lr_min)
    else:
        # Cosine Annealing
        t_cur = epoch - warmup_epochs
        t_max = total_epochs - warmup_epochs
        return lr_min + 0.5 * (lr_max - lr_min) * (1.0 + np.cos((t_cur / t_max) * np.pi))

มาพล็อตตัวกำหนดค่าทั้งสองแบบนี้ตลอด 100 epochs กัน

In [ ]:
epochs = np.arange(100)
lr_step = [step_decay(e) for e in epochs]
lr_cosine = [cosine_annealing_with_warmup(e) for e in epochs]

plt.figure(figsize=(10, 6))
plt.plot(epochs, lr_step, color='purple', linewidth=2.5, label='Step Decay (Gamma=0.5, Step=20)')
plt.plot(epochs, lr_cosine, color='teal', linewidth=3, label='Cosine Annealing with Warmup')
plt.xlabel('Epoch')
plt.ylabel('Learning Rate (lr)')
plt.title('Learning Rate Schedules')
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend()
plt.show()

## 2. การเปรียบเทียบเส้นทางการหาค่าเหมาะที่สุดบน Paraboloid

มาหาค่าเหมาะที่สุด (optimize) ของพาราโบลอยด์แบบ 2 มิติ $f(x, y) = x^2 + 3y^2$ โดยเริ่มต้นที่จุด $(8.0, 8.0)$ โดยใช้:
1.  **Constant Learning Rate ($\alpha = 0.35$):** จะเด้งไปมาและวิ่งเลยจุดต่ำสุด (overshooting)
2.  **Cosine Annealing Learning Rate ($\alpha_{initial} = 0.35$):** เริ่มต้นอย่างรวดเร็ว จากนั้นช้าลง และลู่เข้าหาจุด $(0,0)$ ได้อย่างราบรื่น

In [ ]:
def cost_func(x, y):
    return x**2 + 3 * y**2

def grad_func(x, y):
    return np.array([2*x, 6*y])

def optimize_constant_lr(start_pos, lr=0.32, epochs=30):
    pos = np.array(start_pos, dtype=float)
    history = [pos.copy()]
    for _ in range(epochs):
        grad = grad_func(pos[0], pos[1])
        pos -= lr * grad
        history.append(pos.copy())
    return np.array(history)

def optimize_cosine_lr(start_pos, lr_max=0.32, lr_min=0.001, epochs=30):
    pos = np.array(start_pos, dtype=float)
    history = [pos.copy()]
    for epoch in range(epochs):
        lr = lr_min + 0.5 * (lr_max - lr_min) * (1.0 + np.cos((epoch / epochs) * np.pi))
        grad = grad_func(pos[0], pos[1])
        pos -= lr * grad
        history.append(pos.copy())
    return np.array(history)

start = [8.0, 8.0]
path_const = optimize_constant_lr(start)
path_cosine = optimize_cosine_lr(start)

มาพล็อตทั้งสองเส้นทางนี้บนแผนที่คอนทัวร์แบบ 2 มิติกัน

In [ ]:
x = np.linspace(-10, 10, 100)
y = np.linspace(-10, 10, 100)
X, Y = np.meshgrid(x, y)
Z = cost_func(X, Y)

plt.figure(figsize=(10, 8))
contours = plt.contour(X, Y, Z, levels=20, cmap='viridis')
plt.clabel(contours, inline=1, fontsize=8)

plt.plot(path_const[:, 0], path_const[:, 1], color='red', marker='o', linewidth=1.5, label='Constant LR (Wiggles/Overshoots)')
plt.plot(path_cosine[:, 0], path_cosine[:, 1], color='teal', marker='s', linewidth=2.5, label='Cosine Annealing LR (Stable Convergence)')

plt.scatter(0, 0, color='blue', s=120, marker='*', zorder=5, label='Minimum')
plt.xlabel('x')
plt.ylabel('y')
plt.title('Constant Learning Rate vs. Cosine Annealing')
plt.legend()
plt.show()

จากการสังเกต:
-   **Constant LR (สีแดง):** แกว่งไปมาอย่างรุนแรงตามแกน $y$ เนื่องจากความชันที่สูงมาก ทำให้วิ่งเลยกึ่งกลาง
-   **Cosine Annealing (สีน้ำเงินแกมเขียว):** ขนาดการก้าว (step sizes) ที่ใหญ่ในช่วงแรกช่วยให้เคลื่อนผ่านพื้นที่ได้อย่างรวดเร็ว เมื่อเข้าใกล้กึ่งกลาง ค่า learning rate จะลดลง ช่วยลดการแกว่งและทำให้แบบจำลองลงตัวที่จุดต่ำสุดรวม (global minimum) ได้อย่างสมบูรณ์แบบ

## 💡 การเชื่อมโยงไปยัง YOLO และการเรียนรู้เชิงลึก (Deep Learning)
*   **ไฮเปอร์พารามิเตอร์ของ YOLO:** YOLO กำหนดพารามิเตอร์การวางกำหนดเวลาเหล่านี้ไว้ในการตั้งค่าเริ่มต้น:
    -   `lr0`: อัตราการเรียนรู้เริ่มต้น (เช่น `0.01`)
    -   `lrf`: อัตราส่วนอัตราการเรียนรู้สุดท้าย (เช่น `0.01` ซึ่งหมายความว่า lr สุดท้ายคือ `lr0 * lrf`)
    -   `warmup_epochs`: ระยะเวลาของขั้นตอนการวอร์มอัพแบบเชิงเส้น (ค่าเริ่มต้น `3.0` epochs)
    -   `warmup_bias_lr`: อัตราการเรียนรู้การวอร์มอัพสำหรับ bias (ค่าเริ่มต้น `0.1`)
*   สิ่งนี้ช่วยให้มั่นใจได้ว่าน้ำหนัก (weights) จะไม่ระเบิด (explode) ในช่วงแรกของขั้นตอน Backpropagation และจะลู่เข้าสู่ค่าต่ำสุดระดับท้องถิ่นหรือรวม (local/global minima) ที่เสถียรเมื่อสิ้นสุดการฝึกสอน